# Évaluation

**Objectif** : Construire un classifieur pour prédire si un client va quitter la banque (Exited = 1) ou non (Exited = 0).

## 1. Imports des données

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('Data/Churn_Modelling.csv', sep=';')
df_scoring = pd.read_csv('Data/Churn_Modelling_scoring.csv', sep=';')

print(f"Données d'entraînement : {df.shape}")
print(f"Données de scoring : {df_scoring.shape}")
df.head()

## 2. Exploration des données

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Valeurs manquantes :")
print(df.isnull().sum())

In [ ]:
print("Distribution de la variable cible (Exited) :")
print(df['Exited'].value_counts())
print(f"\nTaux de churn : {df['Exited'].mean():.2%}")

plt.figure(figsize=(6, 4))
df['Exited'].value_counts().plot(kind='bar', color=['steelblue', 'coral'])
plt.title('Distribution de la variable Exited')
plt.xlabel('Exited')
plt.ylabel('Nombre de clients')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['Geography', 'Gender']):
    pd.crosstab(df[col], df['Exited']).plot(kind='bar', ax=ax)
    ax.set_title(f'{col} vs Exited')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), num_cols):
    df.boxplot(column=col, by='Exited', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('Exited')

plt.suptitle('Distribution des variables numériques par classe', y=1.02)
plt.tight_layout()
plt.show()

## 3. Préparation des données

In [ ]:
def prepare_features(df):
    data = df.copy()
    
    cols_to_drop = ['RowNumber', 'CustomerId']
    data = data.drop(columns=[c for c in cols_to_drop if c in data.columns])
    
    data = pd.get_dummies(data, columns=['Geography', 'Gender'], drop_first=True)
    
    return data

df_prepared = prepare_features(df)

X = df_prepared.drop(columns=['Exited'])
y = df_prepared['Exited']

print(f"Features : {X.shape}")
print(f"Colonnes : {list(X.columns)}")

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train : {X_train.shape}, Val : {X_val.shape}")
print(f"Distribution train : {y_train.value_counts().to_dict()}")
print(f"Distribution val : {y_val.value_counts().to_dict()}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

## 4. Définition de la métrique personnalisée

La métrique à optimiser est : `0.5 * Précision(classe 0) + 0.5 * Précision(classe 1)`

In [ ]:
from sklearn.metrics import make_scorer

def custom_score(y_true, y_pred):
    prec_0 = precision_score(y_true, y_pred, pos_label=0)
    prec_1 = precision_score(y_true, y_pred, pos_label=1)
    return 0.5 * prec_0 + 0.5 * prec_1

custom_scorer = make_scorer(custom_score)

## 5. Entraînement et comparaison de modèles

In [ ]:
def evaluate_model(name, model, X_tr, X_v, y_tr, y_v):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_v)
    
    score = custom_score(y_v, y_pred)
    prec_0 = precision_score(y_v, y_pred, pos_label=0)
    prec_1 = precision_score(y_v, y_pred, pos_label=1)
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Précision classe 0 : {prec_0:.4f}")
    print(f"Précision classe 1 : {prec_1:.4f}")
    print(f"Score custom       : {score:.4f}")
    print(f"\n{classification_report(y_v, y_pred)}")
    
    return score, model

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
score_lr, model_lr = evaluate_model("Régression Logistique", lr, X_train_scaled, X_val_scaled, y_train, y_val)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
score_rf, model_rf = evaluate_model("Random Forest", rf, X_train, X_val, y_train, y_val)

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
score_gb, model_gb = evaluate_model("Gradient Boosting", gb, X_train, X_val, y_train, y_val)

In [ ]:
results = pd.DataFrame({
    'Modèle': ['Régression Logistique', 'Random Forest', 'Gradient Boosting'],
    'Score Custom': [score_lr, score_rf, score_gb]
}).sort_values('Score Custom', ascending=False)

print("\nRécapitulatif des scores :")
print(results.to_string(index=False))

## 6. Optimisation du meilleur modèle avec GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'min_samples_split': [2, 5],
    'subsample': [0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    scoring=custom_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nMeilleurs paramètres : {grid_search.best_params_}")
print(f"Meilleur score CV : {grid_search.best_score_:.4f}")

In [ ]:
best_model = grid_search.best_estimator_
score_best, _ = evaluate_model("Gradient Boosting", best_model, X_train, X_val, y_train, y_val)

In [ ]:
y_val_pred = best_model.predict(X_val)

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_val, y_val_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non Churn', 'Churn'], yticklabels=['Non Churn', 'Churn'])
plt.title('Matrice de confusion - Meilleur modèle')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.tight_layout()
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(feature_importance['Feature'], feature_importance['Importance'])
plt.title('Importance des features')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(feature_importance.to_string(index=False))

## 7. Réentraînement sur toutes les données et prédiction sur le scoring

In [ ]:
final_model = GradientBoostingClassifier(**grid_search.best_params_, random_state=42)
final_model.fit(X, y)

In [ ]:
scoring_ids = df_scoring['CustomerId']
df_scoring_prepared = prepare_features(df_scoring)

print(f"Colonnes entraînement : {list(X.columns)}")
print(f"Colonnes scoring      : {list(df_scoring_prepared.columns)}")

df_scoring_prepared = df_scoring_prepared[X.columns]

In [ ]:
predictions = final_model.predict(df_scoring_prepared)

print(f"Nombre de prédictions : {len(predictions)}")
print(f"Distribution des prédictions :")
print(pd.Series(predictions).value_counts())

In [ ]:
X_scoring = pd.DataFrame({
    'CustomerID': scoring_ids,
    'classe': predictions
})

X_scoring.to_csv('Churn_scoring_predictions.csv', index=False, columns=['CustomerID', 'classe'])

print("Fichier 'Churn_scoring_predictions.csv' créé avec succès.")
print(f"\nAperçu :")
X_scoring.head(10)